# 06 Input size on the static Phantom Ph2 images

Grouped cross-validation on the static Phantom images acquired in Ph2, with the same groups as in notebook 05.
Only the input size of the networks changes:

* fully connected network N² → 64 → 64 → 2 for N = 8 (the chip network), 12, 16, 24, 32, 48 and 64,
* CNN for N = 16, 24, 32, 48 and 64 (four 2 × 2 pooling layers need at least 16 × 16 pixels),
* feature control for comparison.

All configurations use the same five folds and three seeds. Every input is computed from the 64 × 64 base image.
The size filter is applied to all static Phantom images (BF and Ph2) before the Ph2 images are selected, as in
notebook 05.

Thesis: Results, Section "Classification of the static Phantom dataset", paragraph "Input size"
(Figure `fig:res-input-size-phantom`, drawn in notebook 08).

Output: `results/06_input_size_ph2_metrics.csv` and `_predictions.csv.gz`.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
from IPython.display import display

import thesis_config as C     # paths and settings
import thesis_ml as M         # data, preprocessing, models, training and evaluation

pd.set_option('display.precision', 3)
C.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Results folder:', C.RESULTS_DIR.resolve())
if C.FAST:
    print('QUICK TEST (THESIS_FAST=1): few images and 3 epochs, the numbers are not meaningful.')

## 1. Settings and data

In [ ]:
MODE = 'Ph2'
DENSE_SIZES = [8, 12, 16, 24, 32, 48, 64]   # N = 8 is the chip network
CNN_SIZES = [16, 24, 32, 48, 64]
G = C.PS_GROUP_COL

PS = C.load_data(sets=['phantom_static'])['phantom_static']
sel = PS.meta['mode'].eq(MODE).to_numpy()
meta, X, feat = PS.meta[sel].reset_index(drop=True), PS.X[sel], PS.feat[sel].reset_index(drop=True)
display(meta.groupby(['date', 'label']).agg(images=('uid', 'size'), raw_images=(G, 'nunique')))

# side of the square crop (the longer crop side), for the axis "µm per input pixel"
side_um = float(np.median(np.maximum(feat.h_um, feat.w_um)))
print(f'Median side of the square crop: {side_um:.1f} µm = {side_um / C.PHANTOM_PX_UM:.0f} pixels '
      f'at {C.PHANTOM_PX_UM} µm/px')

## 2. Cross-validation for every input size

In [ ]:
metrics, preds = [], []
for n in DENSE_SIZES:
    kind = 'ChipNet' if n == 8 else 'Dense'
    m, p, _ = M.run_grouped_cv(meta, X, kind, n, G, C.TRAIN, C.N_FOLDS, C.SEEDS, C.VAL_FRAC,
                               scenario='ps_cv_size', verbose=False)
    metrics.append(m.assign(network='Fully connected')); preds.append(p.assign(network='Fully connected'))
    print(f'fully connected N = {n:2d}: macro-F1 {m.macro_f1.mean():.3f} ± {m.macro_f1.std():.3f}')
for n in CNN_SIZES:
    m, p, _ = M.run_grouped_cv(meta, X, 'CNN', n, G, C.TRAIN, C.N_FOLDS, C.SEEDS, C.VAL_FRAC,
                               scenario='ps_cv_size', verbose=False)
    metrics.append(m.assign(network='CNN')); preds.append(p.assign(network='CNN'))
    print(f'CNN             N = {n:2d}: macro-F1 {m.macro_f1.mean():.3f} ± {m.macro_f1.std():.3f}')
metrics.append(M.run_control_cv(meta, feat, G, C.N_FOLDS, scenario='ps_cv_size').assign(network='Feature control'))
metrics = pd.concat(metrics, ignore_index=True)
preds = pd.concat(preds, ignore_index=True)
metrics['side_um'] = side_um
metrics['um_per_input_px'] = side_um / metrics.input_size.replace(0, np.nan)
M.save_results('06_input_size_ph2', C.RESULTS_DIR, metrics, preds)

## 3. Results
Macro-F1, mean ± standard deviation over folds and seeds.

In [ ]:
table = (metrics.groupby(['network', 'input_size'])
         .agg(um_per_input_px=('um_per_input_px', 'first'), macro_f1=('macro_f1', 'mean'),
              macro_f1_sd=('macro_f1', 'std'), runs=('seed', 'size')))
display(table.round(3))